# 08 — Corpus-anchored supplemental transfer

Dört mevcut model, 13 girdi görünümü, **52 forward /104 cohort skor kümesi**. Eğitim yok. İlk adım T4 /2026.07 runtime üzerinde gerçek readiness; tüm çalıştırma anahtarları Git sürümünde kapalıdır. Eski deneylerin üzerine yazılmaz.

Üç yeni bozulma corpus istatistiklerine dayanır; full folding aynı kelime maskesinde kontrol sağlar. Bu çalışma eski test metinlerinde ek kontrollü değerlendirmedir. Final rapora sonuçlar tamamlanıp denetlendikten sonra eklenecek.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, signal, subprocess
ROOT = Path('/content/cmpe586-transfer-v1')
STORAGE = Path('/content/drive/MyDrive/cmpe586-turkish-punctuation')
TRANSFER = STORAGE / 'preparation/transfer-noise-v1/transfer-v1'
LAUNCH = STORAGE / 'preparation/transfer-noise-v1/launch-2026-11-04'
READINESS = STORAGE / 'preparation/transfer-noise-v1/readiness-v1'
RUN = STORAGE / 'artifacts/evaluations/transfer-noise-v1'
RUN_SETUP = False
RUN_READINESS = False
RUN_TEST = False
BUNDLE_SHA = 'd13f88d06fd97f5a74a891fefa0f33248bb7e9be4f7363063570d57f4905ed80'
RESTORER_SHA = 'e0ba986c3e197afc8b337dc510c588b82e9acfcca26281d4df3564dde30c128a'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONPATH'] = str(ROOT / 'src')
print('Configuration loaded; execution switches are off by default.')


## Runtime ve Drive

T4 ve runtime version **2026.07** seç. Bu hücre mevcut ortamı gösterir ve proje Drive alanını bağlar. Hash/runtime farkları atlanmaz.


In [ ]:
if RUN_SETUP:
    import torch, platform
    print('BASE_RUNTIME', json.dumps({'python':platform.python_version(),'torch':torch.__version__,
        'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),
        'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive', timeout_ms=600_000)
    print('STORAGE_EXISTS', STORAGE.is_dir())
else:
    print('Setup disabled.')


## Hash ile restore

22 MB girdi/kaynak paketi açılır; eski validation ZIP’inden yalnızca doğrulanmış CSV kullanılır. Fine-tuned modeller Drive’da kalır. Soğuk runtime’da published checkpoint sabit revision’dan önbelleğe alınır. Paket sürümleri kayıtlı kısıtlara uymalıdır. Bu adım model forward veya skor üretmez.


In [ ]:
def digest(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def run_step(label, command):
    LAUNCH.mkdir(parents=True, exist_ok=True)
    process = subprocess.Popen([sys.executable, '-u', *map(str, command)], cwd='/content',
        env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True)
    globals()[label + '_process'] = process
    try:
        with (LAUNCH / (label + '.log')).open('a') as log:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
        exit_code = process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        process.wait()
        raise
    if exit_code:
        raise RuntimeError(f'{label} stopped: exit={exit_code}; inspect the saved log.')

if RUN_SETUP:
    helper = TRANSFER / 'restore_transfer_colab.py'
    archive = TRANSFER / 'transfer-execution-v1.zip'
    if digest(helper) != RESTORER_SHA or digest(archive) != BUNDLE_SHA:
        raise ValueError('Restorer or full bundle hash differs; stop.')
    run_step('restoration', [helper, '--archive', archive, '--sha256', BUNDLE_SHA,
        '--project', ROOT, '--storage', STORAGE, '--receipt', LAUNCH / 'restoration.json'])
else:
    print('Restore disabled.')


## Readiness — validation-only

Kaydedilen 13 test girdisi yeniden tokenization/hedef denetiminden geçer. Dört checkpoint’in gerçek dosya/tensor kimlikleri ve sabit runtime doğrulanır. Teknik forward yalnızca validation örnekleri ve validation’dan padding ile oluşturulan maksimum shape üzerindedir; test tahmini/F1 veya optimizer update yoktur.


In [ ]:
if RUN_READINESS:
    run_step('readiness', [ROOT / 'scripts/ready_transfer_evaluation.py',
        '--input-root', ROOT, '--storage-root', STORAGE, '--output', READINESS])
else:
    print('Readiness disabled.')


## Supplemental değerlendirme

Yalnızca readiness geçtikten sonra `RUN_TEST=True`. İlk clean anchor ve noisy koşullar tamamlanınca log’da `TRANSFER_COMMITTED` görünür. Kesintide aynı kaynak/input/model/runtime kimliğiyle tamamlanmış koşullar yeniden kullanılır. Bu hücre değerlendirme boyunca çalışır; Colab sekmesini açık tut.


In [ ]:
if RUN_TEST:
    run_step('evaluation', [ROOT / 'scripts/evaluate_transfer.py', '--input-root', ROOT,
        '--storage-root', STORAGE, '--readiness', READINESS, '--allow-test-scoring'])
else:
    print('Scoring disabled.')


## Durum — salt okunur

Değerlendirme bittiğinde kayıtlı tahminleri denetleyip karşılaştırmaları hesaplayacağız; sonra final raporu güncelleyeceğiz.


In [ ]:
for label, path in [('Readiness', READINESS / 'readiness.json'), ('Evaluation', RUN / 'progress.json')]:
    if path.exists():
        item = json.loads(path.read_text())
        print(label, {k:item[k] for k in ('status','identity','completed_conditions','planned_conditions','last_committed') if k in item})
    else:
        print(label, 'No completed receipt; not assumed to have run.')
